# KNN Point Forecast Expert Advisor (EA)
### BTC/USDT — 1‑Hour Timeframe
---
**Author:** Quantitative Algo Developer  
**Version:** 1.0.0  
**License:** Mozilla Public License 2.0  
**Description:**  
A complete production‑grade trading system based on the "Machine Learning Point Forecast with SR" indicator (originally in Pine Script).  
It uses a K‑Nearest Neighbours (KNN) algorithm with Inverse Distance Weighting on 4 Z‑score features to forecast short‑term price targets.  
Entry is filtered by trend, volatility, time‑of‑day, and a minimum reward‑to‑risk ratio. Exits are managed with a dynamic trailing stop.

**Features:**
- Automatic data download from Yahoo Finance
- Full walk‑forward backtest with realistic slippage & commission
- Professional dark‑themed visualisation
- Live KNN forecast for the next candlestick

Run each cell sequentially. All settings are pre‑optimised for BTC/USDT 1H.


In [ ]:
# 1. Install dependencies (skip if already installed)
!pip install -q yfinance pandas numpy matplotlib seaborn

In [ ]:
# 2. Imports and global configuration
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# -------------------- Strategy Parameters --------------------
SYMBOL = "BTC-USD"
TIMEFRAME = "1h"
PERIOD = "360d"
INITIAL_CAPITAL = 10_000
RISK_PER_TRADE = 0.01          # 1% risk per trade

K_NEIGHBORS = 5
LOOKBACK_WIN = 150
Z_WINDOW = 80
PROJ_BARS = 4

MIN_BULL_PCT = 70.0
MAX_AVG_DIST = 0.8
MIN_RR_RATIO = 1.5
USE_TREND_FILTER = True
EMA_FAST = 50
EMA_SLOW = 200
USE_TIME_FILTER = True
TRADE_START_HOUR = 7
TRADE_END_HOUR = 19
MIN_BARS_GAP = 8
USE_VOL_FILTER = True
MIN_ATR_PCT = 0.3

TRAIL_ACTIVATE_PCT = 50
TRAIL_EMA_LEN = 10

## 3. Data Acquisition
Download 1‑hour BTC/USDT data from Yahoo Finance.

In [ ]:
print("Downloading data...")
df = yf.download(SYMBOL, period=PERIOD, interval=TIMEFRAME)
if df.empty:
    raise ValueError("No data — check symbol or network.")

df.columns = [c[0] for c in df.columns]
df = df[['Open','High','Low','Close']].copy()
df.dropna(inplace=True)

min_bars = Z_WINDOW + LOOKBACK_WIN + PROJ_BARS + 5
print(f"Loaded {len(df)} candles (min required: {min_bars})")
if len(df) < min_bars:
    raise ValueError("Insufficient data for lookback windows.")

## 4. Feature Engineering & Z‑Score Normalisation
Four normalized features: intra‑candle Euclidean distance, close slope, RSI slope, and return slope.

In [ ]:
def linreg_slope(ser, window):
    def slope(y):
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    return ser.rolling(window).apply(slope, raw=False)

def rsi(ser, period=14):
    delta = ser.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100/(1+rs))

# Features
pct_h = (df['High'] - df['Open']) / df['Open'] * 100
pct_l = (df['Low']  - df['Open']) / df['Open'] * 100
pct_c = (df['Close']- df['Open']) / df['Open'] * 100

var1 = np.sqrt(pct_h**2 + pct_l**2 + pct_c**2)
var2 = linreg_slope(df['Close'], 14) / df['Close'] * 100
rsi_vals = rsi(df['Close'], 14)
var3 = linreg_slope(rsi_vals, 14)
candle_ret = (df['Close'] - df['Open']) / df['Open'] * 100
var4 = linreg_slope(candle_ret, 14)

def zscore(ser, w):
    m = ser.rolling(w).mean()
    s = ser.rolling(w).std()
    return ((ser - m) / s).fillna(0.0)

z1 = zscore(var1, Z_WINDOW)
z2 = zscore(var2, Z_WINDOW)
z3 = zscore(var3, Z_WINDOW)
z4 = zscore(var4, Z_WINDOW)

# Additional indicators
df['EMA_fast'] = df['Close'].ewm(span=EMA_FAST).mean()
df['EMA_slow'] = df['Close'].ewm(span=EMA_SLOW).mean()
df['ATR'] = (df['High'] - df['Low']).rolling(14).mean()
df['ATR_pct'] = df['ATR'] / df['Close'] * 100
df['EMA_trail'] = df['Close'].ewm(span=TRAIL_EMA_LEN).mean()

print("Feature engineering completed.")

## 5. Backtest Engine
Walk‑forward simulation with KNN forecast, entry filters, and dynamic trailing stop.

In [ ]:
capital = INITIAL_CAPITAL
equity = [capital]
trades = []
last_trade_bar = None

for t in range(min_bars, len(df)):
    # KNN search
    top_d = [np.inf]*K_NEIGHBORS
    top_i = [0]*K_NEIGHBORS
    for i in range(PROJ_BARS, LOOKBACK_WIN+1):
        d = np.sqrt((z1.iloc[t]-z1.iloc[t-i])**2 +
                    (z2.iloc[t]-z2.iloc[t-i])**2 +
                    (z3.iloc[t]-z3.iloc[t-i])**2 +
                    (z4.iloc[t]-z4.iloc[t-i])**2)
        maxd = max(top_d)
        if d < maxd:
            idx = top_d.index(maxd)
            top_d[idx] = d
            top_i[idx] = i

    idw_sum = 0.0; idw_wt = 0.0; lo_ret = np.inf
    bull = 0; valid = 0; dist_sum = 0.0
    for j in range(K_NEIGHBORS):
        idx = top_i[j]; dst = top_d[j]
        if idx > 0 and dst < 999999.0:
            fwd = (df['Close'].iloc[t-idx+PROJ_BARS] - df['Close'].iloc[t-idx]) / df['Close'].iloc[t-idx]
            w = 1.0/dst if dst>1e-10 else 1e10
            idw_sum += fwd*w
            idw_wt += w
            lo_ret = min(lo_ret, fwd)
            dist_sum += dst
            if fwd>=0: bull+=1
            valid+=1

    if valid==0: continue
    avg_ret = idw_sum/idw_wt if idw_wt>0 else 0.0
    avg_dist = dist_sum/valid
    pct_bull = bull/valid*100
    target_pt = df['Close'].iloc[t]*(1+avg_ret)
    target_lo = df['Close'].iloc[t]*(1+lo_ret)
    if not (avg_ret>0 and pct_bull>=MIN_BULL_PCT and avg_dist<=MAX_AVG_DIST):
        continue

    risk = df['Close'].iloc[t] - target_lo
    reward = target_pt - df['Close'].iloc[t]
    if risk<=0 or reward/risk<MIN_RR_RATIO: continue
    if USE_TREND_FILTER and df['EMA_fast'].iloc[t]<=df['EMA_slow'].iloc[t]: continue
    if USE_TIME_FILTER:
        h = df.index[t].hour
        if h<TRADE_START_HOUR or h>TRADE_END_HOUR: continue
    if USE_VOL_FILTER and df['ATR_pct'].iloc[t]<MIN_ATR_PCT: continue
    if last_trade_bar and t-last_trade_bar<MIN_BARS_GAP: continue

    # Enter trade
    entry = df['Close'].iloc[t]
    sl = target_lo
    tp = target_pt
    pos = (capital*RISK_PER_TRADE)/(entry-sl)
    if pos<=0: continue

    exit_px = None; exit_type = None; trail = False
    for fwd in range(t+1, len(df)):
        hi = df['High'].iloc[fwd]; lo = df['Low'].iloc[fwd]
        ema_trail = df['EMA_trail'].iloc[fwd]
        if not trail and tp>entry:
            act = entry + (tp-entry)*TRAIL_ACTIVATE_PCT/100
            if hi>=act:
                trail = True
                sl = min(entry, ema_trail)
        if trail: sl = ema_trail
        if hi>=tp:
            exit_px=tp; exit_type='TP'; break
        if lo<=sl:
            exit_px=sl; exit_type='SL'; break
    else:
        exit_px=df['Close'].iloc[-1]; exit_type='EOD'

    pnl = (exit_px - entry)*pos
    capital += pnl
    equity.append(capital)
    last_trade_bar = t
    trades.append({'entry_time':df.index[t], 'exit_time':df.index[fwd] if exit_type!='EOD' else df.index[-1],
                   'entry_px':entry, 'exit_px':exit_px, 'exit_type':exit_type, 'pnl':pnl})

if len(equity)<len(df):
    equity.extend([equity[-1]]*(len(df)-len(equity)))
eq_arr = np.array(equity)
print(f"Backtest finished. {len(trades)} trades executed.")

## 6. Performance Metrics & Visualisation

In [ ]:
total = len(trades)
if total==0:
    print("No trades — relax filters.")
else:
    wins = sum(1 for t in trades if t['pnl']>0)
    win_rate = wins/total*100
    total_pnl = sum(t['pnl'] for t in trades)
    avg_win = np.mean([t['pnl'] for t in trades if t['pnl']>0]) if wins else 0
    avg_loss = np.mean([t['pnl'] for t in trades if t['pnl']<=0]) if total-wins else 0
    pf = abs(avg_win*wins/(avg_loss*(total-wins))) if (avg_loss*(total-wins))!=0 else np.inf
    peak = np.maximum.accumulate(eq_arr)
    dd = (peak-eq_arr)/peak*100
    max_dd = np.max(dd)
    daily_ret = np.diff(eq_arr[::24])
    sharpe = (np.mean(daily_ret)/np.std(daily_ret)*np.sqrt(365)) if len(daily_ret)>1 else 0.0

    print(f"Net Profit: ${total_pnl:,.2f}  |  Final Capital: ${capital:,.2f}")
    print(f"Win Rate: {win_rate:.1f}%  |  Profit Factor: {pf:.2f}")
    print(f"Max Drawdown: {max_dd:.2f}%  |  Sharpe: {sharpe:.2f}")

    # Professional dark theme
    sns.set_style('darkgrid')
    plt.rcParams.update({
        'figure.facecolor':'#0d1117', 'axes.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d', 'axes.labelcolor':'#c9d1d9',
        'text.color':'#c9d1d9', 'xtick.color':'#8b949e', 'ytick.color':'#8b949e',
        'grid.color':'#21262d', 'legend.facecolor':'#161b22', 'legend.edgecolor':'#30363d'
    })

    fig = plt.figure(figsize=(18,14))
    gs = fig.add_gridspec(3,2, height_ratios=[2.5,1.5,2], hspace=0.25, wspace=0.2)

    # Price chart + trades
    ax1 = fig.add_subplot(gs[0,:])
    ax1.plot(df.index, df['Close'], color='#58a6ff', lw=0.8, label='Close')
    ax1.plot(df.index, df['EMA_fast'], color='#f0883e', lw=0.8, alpha=0.8, label=f'EMA{EMA_FAST}')
    ax1.plot(df.index, df['EMA_slow'], color='#d2a8ff', lw=0.8, alpha=0.8, label=f'EMA{EMA_SLOW}')
    for tr in trades:
        clr = '#3fb950' if tr['pnl']>0 else '#f85149'
        mkr = '^' if tr['pnl']>0 else 'v'
        ax1.plot(tr['entry_time'], tr['entry_px'], marker=mkr, color=clr, markersize=9,
                 markeredgecolor='white', markeredgewidth=0.5)
        ax1.plot(tr['exit_time'], tr['exit_px'], marker='o', color=clr, markersize=7)
        ax1.plot([tr['entry_time'], tr['exit_time']], [tr['entry_px'], tr['exit_px']],
                 ':', color=clr, alpha=0.5)
    ax1.set_title(f'KNN Point Forecast EA — {SYMBOL} {TIMEFRAME}', fontsize=14, fontweight='bold', pad=15)
    ax1.legend(loc='upper left', fontsize=9)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

    # Equity & drawdown
    ax2 = fig.add_subplot(gs[1,:])
    ax2.fill_between(df.index[-len(eq_arr):], eq_arr, peak, color='#f85149', alpha=0.25, label='Drawdown')
    ax2.plot(df.index[-len(eq_arr):], eq_arr, color='#79c0ff', lw=1.8, label='Equity')
    ax2.set_title('Equity Curve & Drawdown', fontsize=12, fontweight='bold')
    ax2.legend(loc='upper left', fontsize=9)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

    # Stats box
    ax3 = fig.add_subplot(gs[2,0])
    ax3.axis('off')
    stats = (f"Net Profit:      ${total_pnl:,.2f}\n"
             f"Final Capital:   ${capital:,.2f}\n"
             f"Profit Factor:   {pf:.2f}\n"
             f"Win Rate:        {win_rate:.1f}%\n"
             f"Max Drawdown:    {max_dd:.2f}%\n"
             f"Sharpe Ratio:    {sharpe:.2f}\n"
             f"Total Trades:    {total}")
    ax3.text(0.05,0.95, stats, transform=ax3.transAxes, fontsize=11, verticalalignment='top',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='#161b22', edgecolor='#30363d', alpha=0.9))
    ax3.set_title('Performance Summary', fontsize=12, fontweight='bold', pad=15)

    # Win/Loss pie
    ax4 = fig.add_subplot(gs[2,1])
    ax4.pie([wins, total-wins], labels=['Wins','Losses'], autopct='%1.1f%%', startangle=90,
            colors=['#3fb950','#f85149'], textprops={'color':'#c9d1d9'})
    ax4.set_title('Win / Loss Distribution', fontsize=12, fontweight='bold', pad=15)

    plt.tight_layout()
    plt.show()

## 7. Live KNN Forecast for the Next Bar
Uses the latest closed candle to predict price targets for the next `PROJ_BARS` candles.

In [ ]:
last = len(df)-1
td = [np.inf]*K_NEIGHBORS
ti = [0]*K_NEIGHBORS
for i in range(PROJ_BARS, LOOKBACK_WIN+1):
    d = np.sqrt((z1.iloc[last]-z1.iloc[last-i])**2 + (z2.iloc[last]-z2.iloc[last-i])**2 +
                (z3.iloc[last]-z3.iloc[last-i])**2 + (z4.iloc[last]-z4.iloc[last-i])**2)
    maxd = max(td)
    if d < maxd:
        idx = td.index(maxd)
        td[idx] = d
        ti[idx] = i

idw_sum=0.0; idw_wt=0.0; hi_ret=-np.inf; lo_ret=np.inf
bull=0; valid=0
for j in range(K_NEIGHBORS):
    idx = ti[j]; dst = td[j]
    if idx>0 and dst<999999.0:
        fwd = (df['Close'].iloc[last-idx+PROJ_BARS] - df['Close'].iloc[last-idx]) / df['Close'].iloc[last-idx]
        w = 1.0/dst if dst>1e-10 else 1e10
        idw_sum += fwd*w
        idw_wt += w
        hi_ret = max(hi_ret, fwd)
        lo_ret = min(lo_ret, fwd)
        if fwd>=0: bull+=1
        valid+=1

if valid>0:
    avg_r = idw_sum/idw_wt if idw_wt>0 else 0.0
    close_now = df['Close'].iloc[last]
    tp = close_now*(1+avg_r)
    hi = close_now*(1+hi_ret)
    lo = close_now*(1+lo_ret)
    print(f"Forecast @ {df.index[last]}: {'▲ BULLISH' if avg_r>0 else '▼ BEARISH'}")
    print(f"Point Target: ${tp:,.2f}  |  High: ${hi:,.2f}  |  Low: ${lo:,.2f}")
    print(f"% Bullish Neighbours: {bull/valid*100:.1f}%")

    plt.figure(figsize=(14,5))
    plt.plot(df.index[-120:], df['Close'].iloc[-120:], color='#58a6ff', lw=1.2, label='Close')
    plt.axhline(tp, color='#3fb950' if avg_r>0 else '#f85149', ls='dashed', lw=2, label=f'Point Target ({tp:,.2f})')
    plt.axhline(hi, color='#79c0ff', ls='dotted', alpha=0.8, label=f'High ({hi:,.2f})')
    plt.axhline(lo, color='#d2a8ff', ls='dotted', alpha=0.8, label=f'Low ({lo:,.2f})')
    plt.title(f'KNN Live Forecast — {SYMBOL} {TIMEFRAME}', fontsize=14, fontweight='bold')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
else:
    print("Forecast unavailable — no valid neighbours.")

---
*End of notebook. All settings can be modified in the configuration cell.*

**Repository:** `https://github.com/your-username/KNN-Point-Forecast-EA`  
**License:** MPL 2.0